# 06 — Live Analyser
### Real-time inference using FAG-TFT + Anomaly Gate
### Five-state predictive maintenance system
> Run 00 and 04 first to generate all model artifacts.

In [8]:
import pandas as pd
import numpy as np
import pickle
import time
import pyodbc
import torch
import torch.nn as nn
import torch.nn.functional as F
import warnings
warnings.filterwarnings('ignore')
try:
    import winsound
    SOUND_AVAILABLE = True
except ImportError:
    SOUND_AVAILABLE = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {DEVICE}')

✅ Device: cpu


In [9]:
# ── AZURE SQL CONFIGURATION ──────────────────────────────────────────
AZURE_SERVER   = 'kreseakreiotprdsrv.database.windows.net'
AZURE_DATABASE = 'KRESEAKREIOTPRD'
AZURE_USERNAME = 'IOTAdmin'
AZURE_PASSWORD = 'oKuvodump5JNG7dM'
AZURE_TABLE    = 'MBP_ControllerData'

# ── THRESHOLD ────────────────────────────────────────────────────────
# Loaded from ae_thresholds.pkl — override manually here if needed
MANUAL_ANOMALY_THRESHOLD = None

# ── CLASSIFIER CONFIDENCE ────────────────────────────────────────────
CLASSIFIER_CONFIDENCE = 0.75

print('✅ Configuration loaded.')

✅ Configuration loaded.


In [10]:
SOLUTIONS = {
    'Healthy'           : 'Machine operating normally. No maintenance required.',
    'Waveness'          : 'Balance gathering. Adjust feed dog height. Check material feed tension. Inspect roller pressure.',
    'High Foot Pressure': 'Adjust the pressure regulator. Check presser foot spring. Calibrate foot pressure setting.',
    'Skip Stitches/Slip': 'Put correct side of needle to needle hole. Replace the needle. Check blade sharpness. Check needle bar height.',
    'Blade Blunt'       : 'Sharpen the blade. Replace blade if sharpening not possible. Check blade alignment after service.',
}
print(f'✅ Solutions loaded for {len(SOLUTIONS)-1} known breakdowns.')

✅ Solutions loaded for 4 known breakdowns.


#### Load Model Classes (must match 04_FAG_TFT.ipynb)

In [11]:
class GRN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, dropout=0.1):
        super().__init__()
        self.fc1=nn.Linear(input_size, hidden_size); self.fc2=nn.Linear(hidden_size, output_size)
        self.gate_fc=nn.Linear(hidden_size, output_size); self.elu=nn.ELU()
        self.sigmoid=nn.Sigmoid(); self.dropout=nn.Dropout(dropout)
        self.layer_norm=nn.LayerNorm(output_size)
        self.residual=nn.Linear(input_size, output_size) if input_size!=output_size else nn.Identity()
    def forward(self, x):
        h=self.elu(self.fc1(x)); h=self.dropout(h)
        gate=self.sigmoid(self.gate_fc(h)); out=gate*self.fc2(h)
        return self.layer_norm(out+self.residual(x))

class HierarchicalVSN(nn.Module):
    def __init__(self, g1_size, g2_size, g3_size, el_size, hidden_size, dropout=0.1):
        super().__init__()
        self.grn_g1=GRN(g1_size,hidden_size,hidden_size,dropout); self.grn_g2=GRN(g2_size,hidden_size,hidden_size,dropout)
        self.grn_g3=GRN(g3_size,hidden_size,hidden_size,dropout); self.grn_el=GRN(el_size,hidden_size,hidden_size,dropout)
        self.sel_g1=nn.Linear(hidden_size,hidden_size); self.sel_g2=nn.Linear(hidden_size,hidden_size)
        self.sel_g3=nn.Linear(hidden_size,hidden_size); self.sel_el=nn.Linear(hidden_size,hidden_size)
        self.across_grn=GRN(hidden_size*4,hidden_size,4,dropout)
        self.group_merge=nn.Linear(hidden_size*4,hidden_size)
    def forward(self, x_g1, x_g2, x_g3, x_el):
        h1=self.grn_g1(x_g1); h2=self.grn_g2(x_g2); h3=self.grn_g3(x_g3); he=self.grn_el(x_el)
        w1=torch.sigmoid(self.sel_g1(h1)); w2=torch.sigmoid(self.sel_g2(h2))
        w3=torch.sigmoid(self.sel_g3(h3)); we=torch.sigmoid(self.sel_el(he))
        h1,h2,h3,he=w1*h1,w2*h2,w3*h3,we*he
        concat=torch.cat([h1,h2,h3,he],dim=-1)
        g_weights=F.softmax(self.across_grn(concat),dim=-1)
        self.group_weights=g_weights.detach().cpu()
        merged=(g_weights[:,0:1]*h1+g_weights[:,1:2]*h2+g_weights[:,2:3]*h3+g_weights[:,3:4]*he)
        return self.group_merge(concat)+merged

class FAG_TFT(nn.Module):
    def __init__(self, g1_size, g2_size, g3_size, el_size, time_steps, num_classes, hidden_size=64, num_heads=4, dropout=0.1):
        super().__init__()
        self.time_steps=time_steps; self.g1_size=g1_size; self.g2_size=g2_size
        self.g3_size=g3_size; self.el_size=el_size
        self.vsn=HierarchicalVSN(g1_size,g2_size,g3_size,el_size,hidden_size,dropout)
        self.lstm_encoder=nn.LSTM(hidden_size,hidden_size,2,batch_first=True,dropout=dropout)
        self.attention=nn.MultiheadAttention(hidden_size,num_heads,dropout=dropout,batch_first=True)
        self.attn_norm=nn.LayerNorm(hidden_size)
        self.post_attn_grn=GRN(hidden_size,hidden_size,hidden_size,dropout)
        self.classifier=nn.Sequential(nn.Linear(hidden_size,32),nn.ReLU(),nn.Dropout(dropout),nn.Linear(32,num_classes))
    def forward(self, x):
        batch=x.size(0); vsn_out=[]
        for t in range(self.time_steps):
            xt=x[:,t,:]
            g1=xt[:,:self.g1_size]; g2=xt[:,self.g1_size:self.g1_size+self.g2_size]
            g3=xt[:,self.g1_size+self.g2_size:self.g1_size+self.g2_size+self.g3_size]; el=xt[:,self.g1_size+self.g2_size+self.g3_size:]
            vsn_out.append(self.vsn(g1,g2,g3,el))
        vsn_seq=torch.stack(vsn_out,dim=1)
        lstm_out,_=self.lstm_encoder(vsn_seq)
        attn_out,_=self.attention(lstm_out,lstm_out,lstm_out)
        attn_out=self.attn_norm(attn_out+lstm_out)
        out=self.post_attn_grn(attn_out[:,-1,:])
        return self.classifier(out)

class AnomalyGate(nn.Module):
    def __init__(self, time_steps, num_features, num_vib=60, num_elec=6):
        super().__init__()
        self.time_steps=time_steps; self.num_vib=num_vib; self.num_elec=num_elec
        self.enc_vib_conv=nn.Sequential(nn.Conv1d(1,32,kernel_size=3,padding=1),nn.BatchNorm1d(32),nn.ReLU(),nn.MaxPool1d(2))
        enc_vib_size=32*(num_vib//2)
        self.enc_lstm=nn.LSTM(enc_vib_size+num_elec,32,batch_first=True)
        self.bottleneck=nn.Linear(32,16)
        self.dec_lstm=nn.LSTM(16,32,batch_first=True)
        self.dec_out=nn.Linear(32,num_features)
    def encode(self, x):
        batch=x.size(0); vib_in=x[:,:,:self.num_vib]; elec_in=x[:,:,self.num_vib:]
        vib_feats=[]
        for t in range(self.time_steps):
            vt=vib_in[:,t,:].unsqueeze(1); vt=self.enc_vib_conv(vt).view(batch,-1)
            vib_feats.append(vt)
        vib_seq=torch.stack(vib_feats,dim=1)
        merged=torch.cat([vib_seq,elec_in],dim=2)
        _,(h,_)=self.enc_lstm(merged)
        return self.bottleneck(h[-1])
    def forward(self, x):
        encoded=self.encode(x); repeated=encoded.unsqueeze(1).repeat(1,self.time_steps,1)
        dec_out,_=self.dec_lstm(repeated)
        return self.dec_out(dec_out)

print('✅ Model classes defined.')

✅ Model classes defined.


#### Load Trained Models & Artifacts

In [12]:
with open('scaler.pkl','rb') as f:      scaler     = pickle.load(f)
with open('encoder.pkl','rb') as f:     encoder    = pickle.load(f)
with open('tft_config.pkl','rb') as f:  cfg        = pickle.load(f)
with open('ae_thresholds.pkl','rb') as f: thresholds = pickle.load(f)

ANOMALY_THRESHOLD = MANUAL_ANOMALY_THRESHOLD or thresholds['anomaly']
TIME_STEPS        = cfg['time_steps']
ELEC_FEATS        = ['machineVoltageMean','machineVoltageMin','machineVoltageMax',
                      'machineCurrentMean','machineCurrentMin','machineCurrentMax']

gate = AnomalyGate(TIME_STEPS, 66).to(DEVICE)
gate.load_state_dict(torch.load('best_anomaly_gate.pt', map_location=DEVICE))
gate.eval()

tft = FAG_TFT(
    g1_size=cfg['g1_size'], g2_size=cfg['g2_size'],
    g3_size=cfg['g3_size'], el_size=cfg['el_size'],
    time_steps=TIME_STEPS,  num_classes=cfg['num_classes'],
    hidden_size=cfg['hidden_size'], num_heads=cfg['num_heads']
).to(DEVICE)
tft.load_state_dict(torch.load('best_fag_tft.pt', map_location=DEVICE))
tft.eval()

print('✅ Models loaded.')
print(f'   Classes           : {list(encoder.classes_)}')
print(f'   Anomaly threshold : {ANOMALY_THRESHOLD:.6f}')

✅ Models loaded.
   Classes           : ['Blade Blunt', 'Healthy', 'High Foot Pressure', 'Skip Stitches/Slip', 'Waveness']
   Anomaly threshold : 0.752581


#### Feature Extraction & Sound Alert

In [13]:
def extract_features(df):
    vib_records = []
    for val in df['machineVibration']:
        vib_dict = {}
        parts = str(val).split(',')
        try:
            for i in range(0, len(parts)-1, 2):
                f_start = int(parts[i])
                vib_dict[f'{f_start}-{f_start+10}Hz'] = int(parts[i+1])
        except: pass
        vib_records.append(vib_dict)
    expected_vib = [f'{i}-{i+10}Hz' for i in range(10, 610, 10)]
    vib_df  = pd.DataFrame(vib_records).reindex(columns=expected_vib, fill_value=0)
    elec_df = df[ELEC_FEATS].ffill().fillna(0).reset_index(drop=True)
    return pd.concat([vib_df.reset_index(drop=True), elec_df], axis=1)

def alert_sound(level):
    if not SOUND_AVAILABLE: return
    if level == 'CRITICAL':
        for _ in range(3): winsound.Beep(1000, 500)
    elif level == 'WARNING':
        for _ in range(2): winsound.Beep(800, 400)

print('✅ Feature extractor ready.')

✅ Feature extractor ready.


#### Live Monitoring Loop — Five-State System

In [14]:
SELECTED_MACHINE = input('Enter Machine Serial Number (e.g. 0521760): ').strip()

driver   = '{ODBC Driver 17 for SQL Server}'
conn_str = (f'DRIVER={driver};SERVER={AZURE_SERVER};PORT=1433;'
            f'DATABASE={AZURE_DATABASE};UID={AZURE_USERNAME};PWD={AZURE_PASSWORD}')
query    = (f"SELECT TOP {TIME_STEPS} * FROM {AZURE_TABLE} "
            f"WHERE machineSerialNumber = '{SELECTED_MACHINE}' ORDER BY dateTime DESC")

last_processed_time = None

try:
    with pyodbc.connect(conn_str) as conn:
        print(f'\n✅ Connected. Monitoring: {SELECTED_MACHINE}')
        print('='*65)

        while True:
            try:
                df_live = pd.read_sql(query, conn)
                df_live = df_live[df_live['machineVibration'].astype(str).str.startswith('10')]

                # Cold start padding
                while len(df_live) < TIME_STEPS:
                    df_live = pd.concat([df_live.iloc[[0]], df_live], ignore_index=True)
                df_live = df_live.iloc[:TIME_STEPS]

                ts = df_live['dateTime'].iloc[0]
                if ts == last_processed_time:
                    time.sleep(1)
                    continue
                last_processed_time = ts

                sl_time = pd.to_datetime(ts) + pd.Timedelta(hours=5, minutes=30)
                df_window = df_live.iloc[::-1].reset_index(drop=True)

                # Feature extraction
                features = extract_features(df_window)
                X_scaled = scaler.transform(features.values)
                X_input  = torch.FloatTensor(X_scaled).unsqueeze(0).to(DEVICE)  # (1,7,66)

                # ── STAGE 1: ANOMALY GATE ────────────────────────────
                with torch.no_grad():
                    recon    = gate(X_input)
                    ae_error = float(torch.mean(torch.abs(X_input - recon)).cpu())

                # ── STAGE 2: FAG-TFT CLASSIFIER ──────────────────────
                with torch.no_grad():
                    logits     = tft(X_input)
                    probs      = torch.softmax(logits, dim=1).cpu().numpy()[0]
                    cls_idx    = np.argmax(probs)
                    confidence = probs[cls_idx]
                    prediction = encoder.classes_[cls_idx]

                    # Group importance for display
                    gw = tft.vsn.group_weights.numpy()[0]
                    dominant_group = ['10-100Hz','100-300Hz','300-610Hz','Electrical'][np.argmax(gw)]

                # ── STATE DETERMINATION (3-STATE FRAMEWORK) ──────────
                if prediction != 'Healthy' and confidence >= CLASSIFIER_CONFIDENCE:
                    state = 'KNOWN BREAKDOWN DETECTED'; state_num = 2; icon = '🚨'
                    alert_sound('CRITICAL')
                elif ae_error >= ANOMALY_THRESHOLD:
                    state = 'UNKNOWN PROBLEM DETECTED'; state_num = 3; icon = '🔴'
                    alert_sound('CRITICAL')
                else:
                    state = 'HEALTHY'; state_num = 1; icon = '✅'

                # ── OUTPUT ───────────────────────────────────────────
                print(f'\n{"-"*65}')
                current_sl_time = pd.Timestamp.now()
                record_utc_time = pd.to_datetime(ts)
                record_sl_time  = record_utc_time + pd.Timedelta(hours=5, minutes=30)

                print(f'CURRENT TIME (SL)    : {current_sl_time.strftime("%Y-%m-%d %H:%M:%S")}')
                print(f'RECORD TIME (UTC)    : {record_utc_time.strftime("%Y-%m-%d %H:%M:%S")}')
                print(f'RECORD TIME (SL)     : {record_sl_time.strftime("%Y-%m-%d %H:%M:%S")}')
                print(f'STATE                : {icon} {state}')
                print(f'FAG-TFT Prediction   : {prediction} ({confidence*100:.1f}% confidence)')
                print(f'Anomaly Gate Error   : {ae_error:.6f}  [threshold={ANOMALY_THRESHOLD:.4f}]')
                print(f'Dominant Freq Group  : {dominant_group}  (VSN attention)')
                print(f'Group Weights        : G1={gw[0]:.2f} G2={gw[1]:.2f} G3={gw[2]:.2f} Elec={gw[3]:.2f}')
                print('-'*65)

                if state_num == 2:
                    print(f'BREAKDOWN  : {prediction}')
                    print(f'SOLUTION   : {SOLUTIONS.get(prediction, "Contact maintenance supervisor.")}')
                elif state_num == 3:
                    print('BREAKDOWN  : Unknown — not in trained breakdown types')
                    print('ACTION     : Stop machine. Contact maintenance supervisor immediately.')
                else:
                    print('No maintenance required.')

            except Exception as e:
                print(f'⚠️  Loop error: {e}')
            time.sleep(1)

except KeyboardInterrupt:
    print('\n🛑 Monitoring stopped.')
except Exception as e:
    print(f'\n❌ Connection error: {e}')


✅ Connected. Monitoring: 0521760

-----------------------------------------------------------------
CURRENT TIME (SL)    : 2026-03-18 13:43:17
RECORD TIME (UTC)    : 2026-03-18 01:40:51
RECORD TIME (SL)     : 2026-03-18 07:10:51
STATE                : ✅ HEALTHY
FAG-TFT Prediction   : Healthy (25.4% confidence)
Anomaly Gate Error   : 0.271133  [threshold=0.7526]
Dominant Freq Group  : Electrical  (VSN attention)
Group Weights        : G1=0.16 G2=0.11 G3=0.04 Elec=0.69
-----------------------------------------------------------------
No maintenance required.
⚠️  Loop error: Execution failed on sql: SELECT TOP 7 * FROM MBP_ControllerData WHERE machineSerialNumber = '0521760' ORDER BY dateTime DESC
('08S01', '[08S01] [Microsoft][ODBC Driver 17 for SQL Server]TCP Provider: An existing connection was forcibly closed by the remote host.\r\n (10054) (SQLExecDirectW); [08S01] [Microsoft][ODBC Driver 17 for SQL Server]Communication link failure (10054)')
unable to rollback
⚠️  Loop error: Execu